[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/18-beyond-attention.ipynb)

# Module 18: Beyond Attention: Linear Attention and Hybrids

Companion notebook to [Module 18](https://transformers-and-math.drmhse.com/modules/18-beyond-attention) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — Linear attention is a recurrence (section 18.1)
Without the softmax, $\sum_j (q_t \cdot k_j)\, v_j$ can be regrouped into a running state $S_t = \sum_j v_j k_j^\top$.

In [ ]:
T, dk, dv = 50, 16, 16
q, k, v = torch.randn(T, dk), torch.randn(T, dk), torch.randn(T, dv)

direct = torch.stack([sum((q[t] @ k[j]) * v[j] for j in range(t + 1)) for t in range(T)])
S, rec = torch.zeros(dv, dk), []
for t in range(T):
    S = S + torch.outer(v[t], k[t])
    rec.append(S @ q[t])
print('sum and recurrence agree:', torch.allclose(direct, torch.stack(rec), atol=1e-4))
print('state size:', tuple(S.shape), 'whatever the length')

## Part 2 — What the delta rule fixes, and what it costs (section 18.2)
**Overwriting.** Write $n$ key-value pairs, then write a *new* value under every key. Plain linear attention only adds, so it returns a blend of old and new; the delta rule erases what the state returns for a key before writing, so it returns the new value.

In [ ]:
def overwrite(n, rule, d=32):
    keys = F.normalize(torch.randn(n, d), dim=1)
    old, new = torch.randn(n, d), torch.randn(n, d)
    S = torch.zeros(d, d)
    for vals in (old, new):
        for kk, vv in zip(keys, vals):
            write = vv if rule == 'add' else vv - S @ kk          # delta rule with β = 1
            S = S + torch.outer(write, kk)
    out = F.normalize(keys @ S.T, dim=1)
    cos = lambda target: (out * F.normalize(target, dim=1)).sum(1).mean().item()
    return cos(new), cos(old)

for n in (4, 8, 16, 32):
    (an, ao), (dn, do) = overwrite(n, 'add'), overwrite(n, 'delta')
    print(f'{n:3} keys rewritten:  additive recalls new {an:.2f} / old {ao:.2f}    delta rule recalls new {dn:.2f} / old {do:.2f}   (cosine)')

**Capacity.** Now store $n$ *distinct* pairs and try to recall each. With keys that are not perpendicular, each delta-rule write partly erases older items that share its direction, so with many items it remembers mainly the recent ones. Neither rule can hold far more items than the state has dimensions: the fixed-size state is the limit (section 18.3's last paragraph).

In [ ]:
def recall(n, rule, d=32):
    keys = F.normalize(torch.randn(n, d), dim=1)
    vals = torch.randn(n, d)
    S = torch.zeros(d, d)
    for kk, vv in zip(keys, vals):
        S = S + torch.outer(vv if rule == 'add' else vv - S @ kk, kk)
    sim = F.normalize(keys @ S.T, dim=1) @ F.normalize(vals, dim=1).T
    return (sim.argmax(1) == torch.arange(n)).float().mean().item()

for n in (8, 32, 128):
    print(f'{n:4} distinct pairs in a 32×32 state:  additive {recall(n, "add"):5.0%}   delta rule {recall(n, "delta"):5.0%}')

## Part 3 — A selective state-space layer (section 18.3)
The state decays by an amount computed from each input, so the layer can choose what to forget. Run it as a loop (generation) and check against a parallel formulation (training).

In [ ]:
d_state = 8
x = torch.randn(T)
decay = torch.sigmoid(torch.randn(T))            # input-dependent, in (0, 1)
B, C = torch.randn(T, d_state), torch.randn(T, d_state)

h, ys = torch.zeros(d_state), []
for t in range(T):
    h = decay[t] * h + B[t] * x[t]
    ys.append(C[t] @ h)
loop = torch.stack(ys)

# Parallel form: h_t = Σ_j (Π_{i=j+1..t} decay_i) B_j x_j
logcum = torch.cumsum(torch.log(decay), 0)
weights = torch.exp(logcum[:, None] - logcum[None, :]).tril()      # products of decays between j and t
parallel = ((weights @ (B * x[:, None])) * C).sum(1)
print('loop and parallel form agree:', torch.allclose(loop, parallel, atol=1e-4))

## Your turn
1. In Part 2, use a fraction $\beta = 0.5$ in the delta rule (`vv - 0.5 * S @ kk`, and scale the write by 0.5). How do the overwrite and capacity results change?
2. Count the numbers stored by a KV cache for 100,000 tokens of one 128-wide head, and by the 128 × 128 state.